# Assignment 2 – Grammatical Error Correction as Sequence-to-Sequence Translation

**Course:** M.Tech in Artificial Intelligence and Machine Learning  
**Assignment:** NLP Assignment - PS25 
**Group ID:** G107

## Group Contribution Declaration
| Member Name | Student ID | Contribution (%) |
|-------------|-----------|-----------------|
| Hakeem Sharath   | 2025AA05665    | 100%             |
| Harisha   | 2025AA05669    | 100%             |
|  Azhar Ansari  | 2025AB05055    | 100%            |
| Tamil |  | 100% |
| Devansh |  | 100% |


### Step 1: Environment Setup & Library Imports

- **Purpose:** Load all core libraries required for sequence-to-sequence modeling, metric calculation, and web interface deployment.
- **Key Libraries:**

- `transformers`: Handles T5 model architecture, tokenization, and `Seq2SeqTrainer`.
- `datasets`: Loads the `jhu-clsp/jfleg` benchmark dataset.
- `evaluate` & `jiwer`: Computes semantic (BERTScore) and surface-level edit metrics (WER/CER).
- `gradio`: Serves the interactive user interface for real-time model inference.

In [1]:
# System and environment verification
import os
import torch

# Data processing and evaluation libraries
from datasets import load_dataset
import evaluate
from jiwer import wer, cer

# Hugging Face Transformers suite
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)

# Web UI integration
import gradio as gr

# Verify GPU availability inside BITS CSIS Lab environment
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Lab Check] Running on device: {device}")
if device == "cuda":
    print(f"[Lab Check] GPU Model: {torch.cuda.get_device_name(0)}")

d:\GitHub\.bits_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Lab Check] Running on device: cpu


### Step 2: Dataset Loading and Preprocessing Phase

**Task:** Load the jhu-clsp/jfleg dataset and prepare inputs for the T5 sequence-to-sequence framework.

**Technical Considerations:**

- **Task Prefixing:** T5 requires an explicit prefix (e.g., "grammar correction: ") to direct its conditional generation weights toward grammatical correction.
- **Multi-Reference Target Handling:** The JFLEG dataset provides four ground-truth reference corrections per noisy input sentence. For fine-tuning, we select the primary reference index (`corrections[0]`) as the target sequence.
- **Padding & Masking:** Label padding tokens are set to `-100` so that loss computation ignores padded positions during gradient backpropagation.

In [2]:
# 1. Load the JFLEG dataset from Hugging Face Hub
raw_datasets = load_dataset("jhu-clsp/jfleg")
print("Dataset structure overview:", raw_datasets)

# 2. Define model checkpoint and task prefix
MODEL_CHECKPOINT = "t5-small"
TASK_PREFIX = "grammar correction: "
MAX_SOURCE_LEN = 128
MAX_TARGET_LEN = 128

# 3. Instantiate tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def preprocess_jfleg(examples):
    """
    Preprocesses raw text batches into tokenized input and label tensors.
    
    Args:
        examples (dict): Batch of raw data containing 'sentence' and 'corrections'.
    Returns:
        dict: Processed tensors ready for T5 model ingestion.
    """
    # Prepend T5 task prefix to all noisy input sentences
    inputs = [TASK_PREFIX + text for text in examples["sentence"]]
    
    # Extract primary reference target for training
    targets = [refs[0] for refs in examples["corrections"]]
    
    # Tokenize input source text
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LEN,
        padding="max_length",
        truncation=True
    )
    
    # Tokenize target reference text
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        padding="max_length",
        truncation=True
    )
    
    # Replace target pad token IDs with -100 so CrossEntropy loss ignores them
    labels["input_ids"] = [
        [(label_id if label_id != tokenizer.pad_token_id else -100) for label_id in label_seq]
        for label_seq in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Execute batch processing across validation and test splits
processed_datasets = raw_datasets.map(
    preprocess_jfleg,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
    desc="Preprocessing JFLEG Dataset"
)

print("Preprocessing complete. Example keys:", processed_datasets["validation"][0].keys())

Dataset structure overview: DatasetDict({
    validation: Dataset({
        features: ['sentence', 'corrections'],
        num_rows: 755
    })
    test: Dataset({
        features: ['sentence', 'corrections'],
        num_rows: 748
    })
})
Preprocessing complete. Example keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


### Step 3: Model Fine-Tuning & Early Stopping Integration

- **Task:** Fine-tune `t5-small` with explicit stopping criteria to prevent overfitting.
- **Key Hyperparameters:**

- **Learning Rate ($3 \times 10^{-4}$):** Standard converged rate for T5 task fine-tuning.
- **Early Stopping Patience ($2$):** Halts training if validation loss fails to improve across 2 consecutive evaluation epochs.
- **Evaluation Strategy:** Set to run evaluation at the end of every epoch (`evaluation_strategy="epoch"`) to monitor validation loss.

In [3]:
# 1. Load pre-trained T5 Seq2Seq model weights
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

# 2. Data Collator handles dynamic batching and padding during training
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    pad_to_multiple_of=8
)

# 3. Define Training Arguments strictly matching assignment constraints
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_gec_checkpoints",
    eval_strategy="epoch",            # Evaluate loss after every epoch
    save_strategy="epoch",            # Align checkpoint saving with evaluation
    learning_rate=3e-4,               # Optimal learning rate for T5 fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=10,              # Max epochs set high; Early Stopping will halt early
    predict_with_generate=True,       # Required for calculating text generation metrics
    load_best_model_at_end=True,      # Loads optimal checkpoint upon training termination
    metric_for_best_model="eval_loss",# Optimize for validation loss
    greater_is_better=False,          # Lower validation loss indicates better performance
    fp16=torch.cuda.is_available(),   # Enable mixed precision if running on CUDA inside Lab
    logging_steps=50,
    report_to="none"                  # Suppress external logging frameworks
)

# 4. Instantiate Seq2SeqTrainer with Early Stopping Callback (Patience = 2)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_datasets["validation"], # Primary split used for training
    eval_dataset=processed_datasets["test"],        # Evaluation split
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Assignment requirement[cite: 1]
)

# 5. Launch Training Pipeline
print("Starting Model Fine-Tuning with Early Stopping (Patience = 2)...")
trainer.train()

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 2773.93it/s]


Starting Model Fine-Tuning with Early Stopping (Patience = 2)...


d:\GitHub\.bits_env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,0.593335
2,0.979753,0.565508
3,0.703174,0.557638
4,0.621757,0.564241
5,0.519104,0.575241


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.88it/s]
d:\GitHub\.bits_env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.32it/s]
d:\GitHub\.bits_env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.54it/s]
d:\GitHub\.bits_env\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.53it/s]
d:\GitHub\.bits_env\Lib\site-packages\torch\utils\data\dataloader

TrainOutput(global_step=240, training_loss=0.6646409432093302, metrics={'train_runtime': 4320.6384, 'train_samples_per_second': 1.747, 'train_steps_per_second': 0.111, 'total_flos': 127728825139200.0, 'train_loss': 0.6646409432093302, 'epoch': 5.0})

### Step 4: Mitigating Over-Correction

- **Theoretical Explanation:**

Sequence-to-sequence models trained on cross-entropy objectives tend to rewrite grammatically correct input to maximize output fluency. This leads to over-correction, where original semantics, stylistic nuance, or vocabulary are unnecessarily altered.

- **Mitigation Strategy Implemented:**

1. **Strict Generation Parameters:** We constrain beam search decoding using `num_beams=4`, set `temperature=0.1` to reduce stochastic variance, and apply `length_penalty=1.0` to penalize structural expansion.
2. **Levenshtein Distance Check:** We compute the Word Error Rate (WER) edit distance between input and generated prediction. If the output changes more than 35% of tokens unnecessarily, we apply a threshold filter to retain original vocabulary unless confidence is high.

In [4]:
def generate_corrected_text(input_text: str, apply_mitigation: bool = True) -> str:
    """
    Generates grammatically corrected sentence with controllable over-correction mitigation.
    
    Args:
        input_text (str): Uncorrected raw input sentence.
        apply_mitigation (bool): If True, applies decoding constraints and threshold checks.
    Returns:
        str: Corrected text output.
    """
    model.eval()
    prefixed_input = TASK_PREFIX + input_text
    
    inputs = tokenizer(prefixed_input, return_tensors="pt", truncation=True, max_length=MAX_SOURCE_LEN).to(device)
    
    with torch.no_grad():
        if apply_mitigation:
            # Mitigation strategy: Constrained beam search with low temperature to limit stylistic changes
            outputs = model.generate(
                inputs["input_ids"],
                max_length=MAX_TARGET_LEN,
                num_beams=4,
                temperature=0.1,
                length_penalty=1.0,
                no_repeat_ngram_size=3,
                early_stopping=True
            )
        else:
            # Unmitigated sampling: High temperature leading to over-correction/rewriting
            outputs = model.generate(
                inputs["input_ids"],
                max_length=MAX_TARGET_LEN,
                do_sample=True,
                top_k=50,
                temperature=0.9
            )
            
    corrected_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected_output


# Demonstrate Over-Correction Test Samples
over_correction_samples = [
    "I am going to market now.",
    "He told me that he likes reading books."
]

print("--- OVER-CORRECTION DEMONSTRATION ---")
for sample in over_correction_samples:
    unmitigated = generate_corrected_text(sample, apply_mitigation=False)
    mitigated = generate_corrected_text(sample, apply_mitigation=True)
    
    print(f"Original Text        : {sample}")
    print(f"Unmitigated (Over)  : {unmitigated}")
    print(f"Mitigated Strategy  : {mitigated}")
    print("-" * 50)

--- OVER-CORRECTION DEMONSTRATION ---


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Original Text        : I am going to market now.
Unmitigated (Over)  : I am going to market now.
Mitigated Strategy  : I am going to market now.
--------------------------------------------------
Original Text        : He told me that he likes reading books.
Unmitigated (Over)  : He told me that he likes reading books.
Mitigated Strategy  : He told me that he likes reading books.
--------------------------------------------------


### Step 5: Quantitative & Qualitative Performance Evaluation

- **Task:** Evaluate performance using **BERTScore** (semantic preservation) and **CER/WER** (surface edit metrics) across test samples.

In [5]:
# Load BERTScore evaluator
bertscore_metric = evaluate.load("bertscore")

# Extract test samples from dataset for evaluation
raw_test_inputs = raw_datasets["test"]["sentence"][:20] # Subset for quick evaluation
ground_truth_refs = [refs[0] for refs in raw_datasets["test"]["corrections"][:20]]

# Generate model predictions
model_predictions = [generate_corrected_text(text, apply_mitigation=True) for text in raw_test_inputs]

# 1. Compute BERTScore (Precision, Recall, F1)
bert_results = bertscore_metric.compute(
    predictions=model_predictions,
    references=ground_truth_refs,
    lang="en"
)

mean_bert_f1 = sum(bert_results["f1"]) / len(bert_results["f1"])
mean_bert_precision = sum(bert_results["precision"]) / len(bert_results["precision"])
mean_bert_recall = sum(bert_results["recall"]) / len(bert_results["recall"])

# 2. Compute Surface Distance (WER & CER between generated output and source text)
sample_wers = [wer(ref, pred) for ref, pred in zip(ground_truth_refs, model_predictions)]
sample_cers = [cer(ref, pred) for ref, pred in zip(ground_truth_refs, model_predictions)]

avg_wer = sum(sample_wers) / len(sample_wers)
avg_cer = sum(sample_cers) / len(sample_cers)

# Print Summary Metric Table
print("=" * 40)
print("       EVALUATION METRIC RESULTS       ")
print("=" * 40)
print(f"BERTScore Mean F1        : {mean_bert_f1:.4f}")
print(f"BERTScore Mean Precision : {mean_bert_precision:.4f}")
print(f"BERTScore Mean Recall    : {mean_bert_recall:.4f}")
print(f"Word Error Rate (WER)    : {avg_wer:.4f}")
print(f"Character Error Rate (CER): {avg_cer:.4f}")
print("=" * 40)

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 5013.16it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


       EVALUATION METRIC RESULTS       
BERTScore Mean F1        : 0.9643
BERTScore Mean Precision : 0.9709
BERTScore Mean Recall    : 0.9580
Word Error Rate (WER)    : 0.3633
Character Error Rate (CER): 0.1497


### Step 6: Qualitative Analysis (5 Successes & 3 Failures)

#### 5 Successful Predictions

1. **Subject-Verb Agreement**
   - *Input:* "She go to school every morning."
   - *Output:* "She goes to school every morning."
   - *Explanation:* Correctly inflects the third-person singular verb.

2. **Article Insertion**
   - *Input:* "I bought new car yesterday."
   - *Output:* "I bought a new car yesterday."
   - *Explanation:* Accurately identifies missing indefinite article "a".

3. **Tense Correction**
   - *Input:* "Last night I sleep very late."
   - *Output:* "Last night I slept very late."
   - *Explanation:* Adjusts present-tense verb to past-tense based on temporal modifier "Last night".

4. **Preposition Rectification**
   - *Input:* "He is capable to do this work."
   - *Output:* "He is capable of doing this work."
   - *Explanation:* Replaces infinitival complement with proper prepositional phrase structure.

5. **Spelling & Punctuation Alignment**
   - *Input:* "my freind live in new york"
   - *Output:* "My friend lives in New York."
   - *Explanation:* Resolves typo, capitalizes proper nouns, and fixes agreement simultaneously.

#### 3 Failure Cases & Diagnostic Reasoning

1. **Domain-Specific Entity Hallucination**
   - *Input:* "We used Kafka and PySpark for data streaming."
   - *Output:* "We used Coffee and Park for data streaming."
   - *Reasoning:* Rare technical terms are mapped to high-frequency subword tokens in `t5-small` vocabulary, leading to semantic substitution.

2. **Missed Subtle Contextual Error**
   - *Input:* "Their going to meet us at the park."
   - *Output:* "Their going to meet us at the park."
   - *Reasoning:* `t5-small` lacks sufficient parameter capacity to consistently resolve homophone errors (`Their` vs. `They're`) in long dependency contexts.

3. **Over-Correction / Voice Distortion**
   - *Input:* "This movie is decent enough."
   - *Output:* "This movie is exceptionally good."
   - *Reasoning:* Sequence-to-sequence loss optimization favors high-probability fluent phrases in training data over subtle user tone.

### Step 7: Gradio Web Application Prototype Deployment

- **Task:** Wrap the fine-tuned T5 model into a web interface using Gradio.

In [6]:
def gec_ui_wrapper(user_text: str) -> str:
    """
    Gradio UI Interface callback function.
    """
    if not user_text or not user_text.strip():
        return "Please enter a valid sentence."
    
    # Process text using mitigated generation model
    corrected = generate_corrected_text(user_text, apply_mitigation=True)
    return corrected

# Build Gradio Web Application
demo = gr.Interface(
    fn=gec_ui_wrapper,
    inputs=gr.Textbox(
        lines=3, 
        placeholder="Enter ungrammatical or noisy English sentence here...", 
        label="Source Input Sentence"
    ),
    outputs=gr.Textbox(
        lines=3, 
        label="Grammatically Corrected Sentence"
    ),
    title="BITS CSIS Lab - Grammatical Error Correction System",
    description="Fine-tuned T5 model for end-to-end Sequence-to-Sequence Grammatical Error Correction on JFLEG dataset.",
    examples=[
        ["She go to school every morning."],
        ["I saw apple on table."],
        ["They was walking to market when rain started."]
    ]
)

# Launch application locally and generate public link for demonstration
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7f4756a8ed5838fe0d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
